# 07. Balance persistence and sign transitions

Quantify how often each subsector records a positive or negative balance, how long those runs last, and what the empirical one-year sign transition frequencies are.

**Reads**

- `outputs/tables/persistence_summary.csv`
- `outputs/tables/persistence_by_regime.csv`
- `outputs/tables/transition_probabilities.csv`
- `data/processed/fiscal_balances_1977_2025.csv`

**Writes**

- Nothing. All three summaries are persisted by the pipeline.

**Method reference:** `METHODOLOGY.md` section 8

In [ ]:
"""Notebook environment: locate the repository and expose its data layers."""

import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve the repository root from wherever the kernel was started, so the
# notebook works both from the repository root and from the notebooks directory.
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

%matplotlib inline

from portugal_fiscal_balance.analysis import figures

RAW = ROOT / 'data' / 'raw'
INTERIM = ROOT / 'data' / 'interim'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
METRICS = ROOT / 'outputs' / 'metrics'

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

print('repository:', ROOT.name)
print('pipeline outputs present:', (PROCESSED / 'fiscal_balances_1977_2025.csv').exists())

## 1. Sign frequencies and run lengths

Counts are taken over the whole 1977-2025 panel. Longest runs are measured in
consecutive years with the same sign.

In [ ]:
persistence = pd.read_csv(TABLES / 'persistence_summary.csv')
display(persistence.round(3))

In [ ]:
figure = figures.balance_sign_states(pd.read_csv(PROCESSED / 'fiscal_balances_1977_2025.csv'))

## 2. The pooled means describe neither regime

The magnitudes above average across the 1995 splice. The two regimes differ enough
that the pooled figure is not a good description of either one: it lands between
them and corresponds to no observed period.

Sign counts are far more robust to pooling, because a sign does not depend on the
level convention of the vintage. Both are shown per regime below so they are read
on one basis.

Runs are deliberately not recomputed per regime. A run is a property of the
uninterrupted series, and truncating it at a window boundary would report the
length of the window rather than the length of the run.

In [ ]:
regime_persistence = pd.read_csv(TABLES / 'persistence_by_regime.csv')
display(regime_persistence.round(3))

In [ ]:
comparison = (
    regime_persistence.pivot(index='sector', columns='regime', values='mean_balance_pct_gdp')
    .join(persistence.set_index('sector')['mean_balance_pct_gdp'].rename('pooled'))
)
display(comparison.round(3))

## 3. One-year sign transitions

Each row of the persisted table is a `state -> next_state` frequency. Pivoting
gives one transition matrix per subsector, where each row sums to one.

In [ ]:
transitions = pd.read_csv(TABLES / 'transition_probabilities.csv')
matrix = transitions.pivot_table(
    index=['sector', 'state'],
    columns='next_state',
    values='probability',
    fill_value=0.0,
)
display(matrix.round(3))
display(transitions.sort_values(['sector', 'state', 'next_state']).round(3))

## Interpretation limits

1. These are **empirical frequencies over 48 transitions**, not a fitted Markov
   model. No standard errors or stationarity tests are claimed.
2. A state that never occurs in the sample has **no estimated row**, which is
   why some matrices are smaller than three by three.
3. Sign persistence describes the **recorded series**. It is not a forecast and
   carries no implication about future balances.
4. Runs that span 1995 also span the **statistical splice**.
5. **Pooled magnitudes are reported for completeness only.** The regime split is
   the form in which means and medians should be read.

---

[Previous: 06. Year-to-year balance attribution](06_year_to_year_attribution.ipynb) | [Next: 08. Structural mean shifts](08_structural_breaks.ipynb)

Every table shown above is also persisted as CSV, so results can be checked without reading notebook state. To rebuild everything from the bundled raw sources:

```bash
poetry install
make all
```